# Homework 2: BERT on AWS
## Nick Visuthikosol

In [30]:
!pip install boto3

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [31]:
!pip install datarec-lib

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [32]:
import boto3

s3 = boto3.client("s3")
print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': 'V6MENGNXM4YS5YTF', 'HostId': 'IpSWA+Q+KsGl1QR/QKLwJ1TYGS/LE3OlJZsSaPCUIKB124mRN2v3p7lL0e+TCndpzDs/gRntn68=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'IpSWA+Q+KsGl1QR/QKLwJ1TYGS/LE3OlJZsSaPCUIKB124mRN2v3p7lL0e+TCndpzDs/gRntn68=', 'x-amz-request-id': 'V6MENGNXM4YS5YTF', 'date': 'Tue, 05 May 2026 21:58:12 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'acharya-cui-sokolenko-mwaa', 'CreationDate': datetime.datetime(2026, 2, 24, 19, 9, 1, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-cui-sokolenko-mwaa'}, {'Name': 'acharya-de300-wi26', 'CreationDate': datetime.datetime(2026, 2, 17, 6, 21, 19, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-de300-wi26'}, {'Name': 'adams-agrawal-evensen-mwaa', 'CreationDate': datetime.datetime(2025, 6, 7, 20, 58, 20, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::adams-agrawal-evensen-mwaa'}, {'Name': 'ads-de300win

In [33]:
BUCKET_NAME          = "data-eng300-hw2-visuthikosol"
EMBEDDINGS_KEY       = "embeddings/embeddings.pkl"
FULL_EMBEDDINGS_KEY  = "embeddings/full_embeddings.pkl"  
DATA_DIR   = "./data"

In [34]:
import os

In [35]:
import os, re, zipfile, requests, pickle, datetime
import numpy as np
import pandas as pd
import boto3
from botocore.exceptions import ClientError
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


## Task 1 — Download MovieLens 1M and Upload to S3

In [38]:
# download zip
os.makedirs(DATA_FOLDER, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading MovieLens 1M...")
    r = requests.get(MOVIELENS_URL)
    with open(ZIP_PATH, "wb") as f:
        f.write(r.content)
    print("Download complete.")
try:
    s3.head_object(Bucket=BUCKET_NAME, Key=DATASET_S3_KEY)
    print("Dataset already in S3, skipping upload.")
except:
    s3.upload_file(ZIP_PATH, BUCKET_NAME, DATASET_S3_KEY)
    print("Uploaded to S3.")

Dataset already in S3, skipping upload.


In [37]:
MOVIELENS_URL  = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
DATA_FOLDER    = "data"
ZIP_PATH       = "data/ml-1m.zip"
DATASET_S3_KEY = "homework_2/data/ml-1m.zip"

In [39]:
task1_download_and_upload_dataset()

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATA_FOLDER)

movies = pd.read_csv(f"{DATA_FOLDER}/ml-1m/movies.dat",
                     sep="::", engine="python",
                     names=["MovieID", "Title", "Genres"],
                     encoding="latin-1")

ratings = pd.read_csv(f"{DATA_FOLDER}/ml-1m/ratings.dat",
                      sep="::", engine="python",
                      names=["UserID", "MovieID", "Rating", "Timestamp"],
                      encoding="latin-1")

users = pd.read_csv(f"{DATA_FOLDER}/ml-1m/users.dat",
                    sep="::", engine="python",
                    names=["UserID", "Gender", "Age", "Occupation", "Zip"],
                    encoding="latin-1")

print(f"Movies: {len(movies)}, Ratings: {len(ratings)}, Users: {len(users)}")

Dataset already exists at s3://data-eng300-hw2-visuthikosol/homework_2/data/ml-1m.zip. Skipping download.
Movies: 3883, Ratings: 1000209, Users: 6040


In [40]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder    = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Loaded {MODEL_NAME}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded distilbert-base-uncased


## Task 2: BERT embeddings

In [43]:
def extract_year(title):
    m = re.search(r"\((\d{4})\)", title)
    return int(m.group(1)) if m else 9999

In [44]:
@torch.no_grad()
def bert_embed(texts, max_len=128):
    encoder.eval()
    batch = tokenizer(texts, padding=True, truncation=True,
                      max_length=max_len, return_tensors="pt")
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out   = encoder(**batch)
    cls   = out.last_hidden_state[:, 0]
    return F.normalize(cls, dim=-1).cpu().numpy().astype("float32")

# pre-1980 movies
df_pre = movies.copy()
df_pre["Year"] = df_pre["Title"].apply(extract_year)
df_pre = df_pre[df_pre["Year"] <= 1980].reset_index(drop=True)
df_pre["text"] = df_pre["Title"] + ". " + df_pre["Genres"].str.replace("|", " ", regex=False)
print(f"Pre-1980 movies: {len(df_pre)}")


Pre-1980 movies: 887


In [45]:
# encode or load from S3
try:
    s3.head_object(Bucket=BUCKET_NAME, Key=EMBEDDINGS_KEY)
    print("Pre-1980 embeddings found in S3, loading...")
    s3.download_file(BUCKET_NAME, EMBEDDINGS_KEY, "embeddings.pkl")
    with open("embeddings.pkl", "rb") as f:
        saved = pickle.load(f)
    vecs_pre = saved["embeddings"]
    df_pre   = saved["movies"]
except:
    print("Computing pre-1980 embeddings...")
    all_vecs = []
    for i in tqdm(range(0, len(df_pre), 32), desc="encoding"):
        all_vecs.append(bert_embed(df_pre["text"].tolist()[i:i+32]))
    vecs_pre = np.vstack(all_vecs)
    with open("embeddings.pkl", "wb") as f:
        pickle.dump({"embeddings": vecs_pre, "movies": df_pre}, f)
    s3.upload_file("embeddings.pkl", BUCKET_NAME, EMBEDDINGS_KEY)
    print("Uploaded pre-1980 embeddings to S3.")

index_pre = faiss.IndexFlatIP(vecs_pre.shape[1])
index_pre.add(vecs_pre)
print(f"Pre-1980 FAISS index: {index_pre.ntotal} items")

Computing pre-1980 embeddings...


encoding:   0%|          | 0/28 [00:00<?, ?it/s]

Uploaded pre-1980 embeddings to S3.
Pre-1980 FAISS index: 887 items


## Task 3: Recommendations for pre-1980

In [46]:
def get_recs(user_text, movies_df, index, seen=set(), k=5):
    u = bert_embed([user_text])
    scores, idx = index.search(u, k + len(seen) + 10)
    recs = []
    for j, score in zip(idx[0], scores[0]):
        if j < 0 or j >= len(movies_df):
            continue
        movie_id = movies_df["MovieID"].tolist()[j]
        if movie_id not in seen:
            row = movies_df[movies_df["MovieID"] == movie_id].iloc[0]
            recs.append({"Title": row["Title"], "Genres": row["Genres"],
                         "score": round(float(score), 4)})
        if len(recs) == k:
            break
    return pd.DataFrame(recs)

In [47]:
# cold user with no history
cold_recs = get_recs("no history", df_pre, index_pre)
print("Cold user recommendations:")
print(cold_recs)
cold_recs.to_csv("cold_user_pre1980.csv", index=False)
s3.upload_file("cold_user_pre1980.csv", BUCKET_NAME, "recommendations/pre1980/cold_user.csv")

# top user with top 5% by rating count
rating_counts = ratings.groupby("UserID").size()
threshold     = rating_counts.quantile(0.95)
top_users     = rating_counts[rating_counts >= threshold].index.tolist()
np.random.seed(42)
top_user_id   = int(np.random.choice(top_users))
print(f"\nTop user: {top_user_id} ({rating_counts[top_user_id]} ratings)")

Cold user recommendations:
                            Title         Genres   score
0         Last Detail, The (1973)   Comedy|Drama  0.9518
1               Champ, The (1979)          Drama  0.9507
2        Long Goodbye, The (1973)          Crime  0.9500
3        Conversation, The (1974)  Drama|Mystery  0.9498
4  Gods Must Be Crazy, The (1980)         Comedy  0.9481

Top user: 1764 (613 ratings)


In [48]:
# get last 3 movies the top user watched
hist = (ratings[(ratings["UserID"] == top_user_id) &
                (ratings["MovieID"].isin(df_pre["MovieID"]))]
        .sort_values("Timestamp").tail(3)["MovieID"].tolist())
user_text = " ".join(df_pre.set_index("MovieID").loc[hist, "text"].tolist()) if hist else "no history"

top_recs = get_recs(user_text, df_pre, index_pre, seen=set(hist))
print("Top user recommendations:")
print(top_recs)
top_recs.to_csv("top_user_pre1980.csv", index=False)
s3.upload_file("top_user_pre1980.csv", BUCKET_NAME, "recommendations/pre1980/top_user.csv")

Top user recommendations:
                               Title                       Genres   score
0                 Blue Hawaii (1961)               Comedy|Musical  0.9680
1          Herbie Rides Again (1974)  Adventure|Children's|Comedy  0.9638
2                      Popeye (1980)     Adventure|Comedy|Musical  0.9634
3  Herbie Goes to Monte Carlo (1977)  Adventure|Children's|Comedy  0.9622
4              Anchors Aweigh (1945)               Comedy|Musical  0.9596


## Task 4: Full dataset

In [51]:
# full catalogue
df_full = movies.copy()
df_full["text"] = df_full["Title"] + ". " + df_full["Genres"].str.replace("|", " ", regex=False)
print(f"Full catalogue: {len(df_full)} movies")

Full catalogue: 3883 movies


In [52]:
# encode or load from S3
try:
    s3.head_object(Bucket=BUCKET_NAME, Key=FULL_EMBEDDINGS_KEY)
    s3.download_file(BUCKET_NAME, FULL_EMBEDDINGS_KEY, "full_embeddings.pkl")
    with open("full_embeddings.pkl", "rb") as f:
        saved = pickle.load(f)
    vecs_full = saved["embeddings"]
    df_full   = saved["movies"]
except:
    all_vecs = []
    for i in tqdm(range(0, len(df_full), 32), desc="encoding"):
        all_vecs.append(bert_embed(df_full["text"].tolist()[i:i+32]))
    vecs_full = np.vstack(all_vecs)
    with open("full_embeddings.pkl", "wb") as f:
        pickle.dump({"embeddings": vecs_full, "movies": df_full}, f)
    s3.upload_file("full_embeddings.pkl", BUCKET_NAME, FULL_EMBEDDINGS_KEY)
    print("Uploaded full embeddings to S3.")

index_full = faiss.IndexFlatIP(vecs_full.shape[1])
index_full.add(vecs_full)
print(f"Full FAISS index: {index_full.ntotal} items")

encoding:   0%|          | 0/122 [00:00<?, ?it/s]

Uploaded full embeddings to S3.
Full FAISS index: 3883 items


In [53]:
# cold user
cold_recs_full = get_recs("no history", df_full, index_full)
print("Cold user recommendations (full):")
print(cold_recs_full)
cold_recs_full.to_csv("cold_user_full.csv", index=False)
s3.upload_file("cold_user_full.csv", BUCKET_NAME, "recommendations/full/cold_user.csv")

# top user
hist_full = (ratings[(ratings["UserID"] == top_user_id) &
                     (ratings["MovieID"].isin(df_full["MovieID"]))]
             .sort_values("Timestamp").tail(3)["MovieID"].tolist())
user_text_full = " ".join(df_full.set_index("MovieID").loc[hist_full, "text"].tolist()) if hist_full else "no history"

top_recs_full = get_recs(user_text_full, df_full, index_full, seen=set(hist_full))
print("Top user recommendations (full):")
print(top_recs_full)
top_recs_full.to_csv("top_user_full.csv", index=False)
s3.upload_file("top_user_full.csv", BUCKET_NAME, "recommendations/full/top_user.csv")

Cold user recommendations (full):
                   Title          Genres   score
0     Source, The (1999)     Documentary  0.9706
1     Cruise, The (1998)     Documentary  0.9613
2  Contender, The (2000)  Drama|Thriller  0.9608
3  Minus Man, The (1999)   Drama|Mystery  0.9607
4       Committed (2000)    Comedy|Drama  0.9595
Top user recommendations (full):
                                               Title  \
0        Police Academy 4: Citizens on Patrol (1987)   
1   Police Academy 5: Assignment: Miami Beach (1988)   
2  Toxic Avenger Part III: The Last Temptation of...   
3                       2001: A Space Odyssey (1968)   
4          Police Academy 3: Back in Training (1986)   

                          Genres   score  
0                         Comedy  0.9770  
1                         Comedy  0.9751  
2                  Comedy|Horror  0.9718  
3  Drama|Mystery|Sci-Fi|Thriller  0.9698  
4                         Comedy  0.9696  


## Task 5: My ratings

In [61]:
movies[movies["Title"].str.contains("Pulp Fiction", case=False)]

,MovieID,Title,Genres
293,296,Pulp Fiction (1994),Crime|Drama


In [62]:
movies[movies["Title"].str.contains("Forrest Gump", case=False)]

,MovieID,Title,Genres
352,356,Forrest Gump (1994),Comedy|Romance|War


In [63]:
movies[movies["Title"].str.contains("Toy Story", case=False)]

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy


In [74]:
movies[movies["Title"].str.contains("Before Sunrise", case=False)]

,MovieID,Title,Genres
213,215,Before Sunrise (1995),Drama|Romance


In [76]:
movies[movies["Title"].str.contains("Taxi Driver", case=False)]


,MovieID,Title,Genres
109,111,Taxi Driver (1976),Drama|Thriller


In [80]:
movies[movies["Title"].str.contains("Blade Runner", case=False)]

,MovieID,Title,Genres
537,541,Blade Runner (1982),Film-Noir|Sci-Fi


In [83]:
movies[movies["Title"].str.contains("When Harry Met Sally", case=False)]

,MovieID,Title,Genres
1287,1307,When Harry Met Sally... (1989),Comedy|Romance


In [87]:
movies[movies["Title"].str.contains("Back to the", case=False)]

,MovieID,Title,Genres
1250,1270,Back to the Future (1985),Comedy|Sci-Fi
1794,1863,Major League: Back to the Minors (1998),Comedy
1942,2011,Back to the Future Part II (1989),Comedy|Sci-Fi
1943,2012,Back to the Future Part III (1990),Comedy|Sci-Fi|Western


In [89]:
MY_RATINGS = {
    # Pulp Fiction (1994)
    296:5, 
    # Blade Runner (1982)
    541:5,   
    # Taxi Driver (1976)
    111:5,   
    # Silence of the Lambs, The (1991)
    593:4, 
    # Toy Story 2 (1999)
    480:4,  
    # Toy Story (1995)
    1:4,  
    # Forrest Gump (1994)
    356:5,
    # Back to the Future (1985)
    1270:4,   
    # Back to the Future Part II (1989)
    2011:4,   
    # Back to the Future Part III (1990)
    2012:2,   
}

assert len(MY_RATINGS) == 10, "Need exactly 10 ratings."


In [90]:
# build user text
#repeat 5-star movies twice
rated_df = df_full[df_full["MovieID"].isin(MY_RATINGS)].copy()
rated_df["My_Rating"] = rated_df["MovieID"].map(MY_RATINGS)

texts = []

In [91]:
for _, row in rated_df.iterrows():
    repeat = 2 if row["My_Rating"] >= 5 else 1
    texts.extend([row["text"]] * repeat)
my_user_text = " ".join(texts)

In [92]:
# recommend
my_recs = get_recs(my_user_text, df_full, index_full, seen=set(MY_RATINGS.keys()))
print("Your 5 recommendations:")
print(my_recs)

Your 5 recommendations:
                                               Title  \
0                Transformers: The Movie, The (1986)   
1                      Santa Claus: The Movie (1985)   
2  Adventures of Buckaroo Bonzai Across the 8th D...   
3          Puppet Master 5: The Final Chapter (1994)   
4     Mystery Science Theater 3000: The Movie (1996)   

                                            Genres   score  
0  Action|Animation|Children's|Sci-Fi|Thriller|War  0.9601  
1                     Adventure|Children's|Fantasy  0.9578  
2                          Adventure|Comedy|Sci-Fi  0.9572  
3                           Horror|Sci-Fi|Thriller  0.9565  
4                                    Comedy|Sci-Fi  0.9558  


In [93]:
# save to S3
rated_df[["MovieID", "Title", "Genres", "My_Rating"]].to_csv("my_ratings.csv", index=False)
s3.upload_file("my_ratings.csv", BUCKET_NAME, "user_profile/my_ratings.csv")

my_recs.to_csv("my_recommendations.csv", index=False)
s3.upload_file("my_recommendations.csv", BUCKET_NAME, "recommendations/full/my_recommendations.csv")